In [ ]:
#pip install requests pandas

In [ ]:
#pip install --upgrade requests urllib3 six
#!/usr/bin/env python3

In [ ]:
import argparse, json, os, random, re, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import requests

DATA_DIR = "tiki_data"
os.makedirs(DATA_DIR, exist_ok=True)


CATEGORIES = {
    1789: "Dien thoai - May tinh bang",
    1846: "Laptop - May vi tinh",
    1815: "Thiet bi so - Phu kien so",
    1882: "Dien gia dung",
    1883: "Nha cua - Doi song",
    8322: "Nha sach Tiki",
    2549: "Do choi - Me & be",
    915:  "Lam dep - Suc khoe",
    1975: "The thao - Da ngoai",
    8594: "O to - Xe may - Xe dap",
}

SORTS = ["top_seller", "newest"]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "vi-VN,vi;q=0.9",
    "Referer": "https://tiki.vn/",
}
_local = threading.local()
NEWEST_PAGES = 3   

def session():
    """Moi luong dung 1 Session rieng (an toan khi chay song song)."""
    if not hasattr(_local, "s"):
        _local.s = requests.Session()
        _local.s.headers.update(HEADERS)
    return _local.s

def get_json(url, params=None, retries=5):
    for i in range(retries):
        try:
            r = session().get(url, params=params, timeout=20)
            if r.status_code == 200:
                time.sleep(random.uniform(0.3, 0.8))
                return r.json()
            if r.status_code == 404:
                return None
            if r.status_code in (429, 403, 500, 502, 503):
                wait = 5 * (2 ** i)
                print(f"  [{r.status_code}] cho {wait}s roi thu lai ...")
                time.sleep(wait)
                continue
            return None
        except (requests.RequestException, ValueError):
            time.sleep(2 * (i + 1))
    return None


def path(name):
    return os.path.join(DATA_DIR, name)


def append_jsonl(fname, rows):
    with open(path(fname), "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def read_jsonl(fname):
    if not os.path.exists(path(fname)):
        return []
    with open(path(fname), encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def run_parallel(items, fn, sink, workers, label):
    n_ok = n_fail = 0
    ex = ThreadPoolExecutor(max_workers=workers)
    try:
        futs = [ex.submit(fn, it) for it in items]
        for i, f in enumerate(as_completed(futs), 1):
            try:
                res = f.result()
            except Exception as e:
                res = None
                print("  loi:", repr(e)[:120])
            if res is None:
                n_fail += 1         
            else:
                sink(res); n_ok += 1
            if i % 50 == 0 or i == len(futs):
                print(f"  {label}: {i}/{len(futs)} (ok={n_ok}, loi={n_fail})")
    except KeyboardInterrupt:
        ex.shutdown(wait=False, cancel_futures=True)
        print("Da dung. Chay lai cung lenh de tiep tuc.")
        raise
    ex.shutdown(wait=True)
    if n_fail:
        print(f"  {label}: {n_fail} muc loi, chay lai lenh de thu lai.")


def strip_html(s):
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", s or "")).strip()


def flatten_scalars(d, prefix=""):
    """Lay moi gia tri don (str/int/float/bool) cua dict, de khong phu thuoc ten field cu the."""
    out = {}
    if not isinstance(d, dict):
        return out
    for k, v in d.items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            out[f"{prefix}{k}"] = v
    return out



def step_listing(pages, limit, newest_pages=NEWEST_PAGES):
    done = {(r["category_id"], r["sort"], r["page"]) for r in read_jsonl("listing_pages.jsonl")}
    for cid, cname in CATEGORIES.items():
        for sort in SORTS:
            max_p = pages if sort == "top_seller" else min(pages, newest_pages)
            for p in range(1, max_p + 1):
                if (cid, sort, p) in done:
                    continue
                js = get_json("https://tiki.vn/api/v2/products", {
                    "limit": 48, "include": "advertisement", "aggregations": 2,
                    "category": cid, "page": p, "sort": sort})
                if js is None:       
                    print(f"  {cname} | {sort} | trang {p}: LOI REQUEST (se thu lai lan chay sau)")
                    break
                items = js.get("data") or []
                append_jsonl("listing_pages.jsonl", [{"category_id": cid, "sort": sort, "page": p}])
                if not items:
                    print(f"  {cname} | {sort} | het du lieu o trang {p}")
                    break
                rows = []
                for it in items:
                    price = it.get("price")
                    lp = it.get("list_price") or it.get("original_price") or price
                    dr = it.get("discount_rate")
                    if dr is None and lp:
                        dr = round((lp - (price or 0)) / lp * 100)
                    qs = it.get("quantity_sold")
                    qs = qs.get("value") if isinstance(qs, dict) else qs
                    m = re.search(r"spid=(\d+)", it.get("url_path") or "")
                    spid = it.get("seller_product_id") or (m.group(1) if m else None)
                    rows.append({
                        "product_id": it.get("id"),
                        "name": it.get("name"),
                        "price": price,
                        "list_price": lp,
                        "discount_rate": dr,
                        "rating_average": it.get("rating_average"),
                        "review_count": it.get("review_count"),
                        "quantity_sold": qs,
                        "brand_name": it.get("brand_name"),
                        "seller_id": it.get("seller_id"),
                        "seller_name": it.get("seller_name"),
                        "seller_product_id": spid,
                        "inventory_status": it.get("inventory_status") or it.get("availability"),
                        "listing_category_id": cid,
                        "listing_category": cname,
                        "listing_sort": sort,
                        "scraped_at": datetime.now().isoformat(timespec="seconds"),
                    })
                append_jsonl("listing.jsonl", rows)
                print(f"  {cname} | {sort} | trang {p}: +{len(rows)}")
  
    seen, uniq = set(), []
    for r in read_jsonl("listing.jsonl"):
        if r["product_id"] not in seen:
            seen.add(r["product_id"])
            uniq.append(r)
    print(f"Tong san pham duy nhat: {len(uniq)}")
    return uniq


def unique_products():
    seen, uniq = set(), []
    for r in read_jsonl("listing.jsonl"):
        if r["product_id"] not in seen:
            seen.add(r["product_id"])
            uniq.append(r)
    return uniq



def fetch_detail(p):
    pid = p["product_id"]
    js = get_json(f"https://tiki.vn/api/v2/products/{pid}",
                  {"platform": "web", "spid": p.get("seller_product_id")})
    if not js:
        return None
    cats = js.get("categories") or {}
    crumbs = [b.get("name") for b in (js.get("breadcrumbs") or []) if b.get("name")]
    specs = []
    for g in js.get("specifications") or []:
        for a in g.get("attributes") or []:
            specs.append({"group": g.get("name"), "name": a.get("name"),
                          "value": strip_html(str(a.get("value")))})
    cs = js.get("current_seller") or {}
    atq = js.get("all_time_quantity_sold")
    return {
        "product_id": pid, "ok": True,
        "leaf_category_id": cats.get("id"), "leaf_category": cats.get("name"),
        "category_path": " > ".join(crumbs),
        "day_ago_created": js.get("day_ago_created"),
        "all_time_quantity_sold": atq.get("value") if isinstance(atq, dict) else atq,
        "stock_qty": (js.get("stock_item") or {}).get("qty"),
        "n_images": len(js.get("images") or []),
        "short_desc_len": len(strip_html(js.get("short_description"))),
        "desc_len": len(strip_html(js.get("description"))),
        "n_warranty_items": len(js.get("warranty_info") or []),
        "has_video": bool(js.get("video_url")),
        "n_specs": len(specs),
        "seller_store_id": cs.get("store_id"),
        "specs": specs,
    }


def step_detail(limit, workers):
    done = {r["product_id"] for r in read_jsonl("detail.jsonl")}
    todo = [p for p in unique_products() if p["product_id"] not in done]
    if limit:
        todo = todo[:limit]
    print(f"Detail: {len(todo)} san pham, {workers} luong")
    run_parallel(todo, fetch_detail, lambda r: append_jsonl("detail.jsonl", [r]), workers, "detail")



def fetch_seller(item):
    sid, p = item
    js = get_json("https://api.tiki.vn/product-detail/v2/widgets/seller",
                  {"seller_id": sid, "mpid": p["product_id"], "spid": p.get("seller_product_id")})
    if js is None:
        return None
    data = js.get("data") or {}
    seller = data.get("seller") or {}
    row = {"seller_id": sid, "seller_name": p.get("seller_name"), "ok": bool(seller)}
    row.update(flatten_scalars(seller, "s_"))
    row.update(flatten_scalars({k: v for k, v in data.items() if k != "seller"}, "w_"))
    return row


def step_sellers(limit, workers):
    rep = {} 
    for p in unique_products():
        rep.setdefault(p["seller_id"], p)
    done = {r["seller_id"] for r in read_jsonl("sellers.jsonl")}
    todo = [(sid, p) for sid, p in rep.items() if sid not in done]
    if limit:
        todo = todo[:limit]
    print(f"Sellers: {len(todo)} nguoi ban, {workers} luong")
    run_parallel(todo, fetch_seller, lambda r: append_jsonl("sellers.jsonl", [r]), workers, "sellers")



def fetch_reviews(item):
    p, max_pages = item
    pid, rows = p["product_id"], []
    for page in range(1, max_pages + 1):
        js = get_json("https://tiki.vn/api/v2/reviews", {
            "product_id": pid, "limit": 20, "page": page,
            "sort": "score|desc,id|desc,stars|all",
            "spid": p.get("seller_product_id"),
            "include": "comments,contribute_info,attributes"})
        if js is None:
            if page == 1:
                return None          
            break
        data = js.get("data") or []
        for r in data:
            cb = r.get("created_by") or {}
            rows.append({
                "review_id": r.get("id"), "product_id": pid,
                "rating": r.get("rating"), "title": r.get("title"),
                "content": r.get("content"), "thank_count": r.get("thank_count"),
                "is_buyer": cb.get("purchased"), "customer_id": cb.get("id"),
                "created_at": r.get("created_at"),
                "n_images": len(r.get("images") or []),
                "n_comments": len(r.get("comments") or []),
            })
        last = (js.get("paging") or {}).get("last_page", page)
        if not data or page >= last:
            break
    return {"product_id": pid, "rows": rows}


def step_reviews(max_pages, limit, workers):
    prods = [p for p in unique_products() if (p.get("review_count") or 0) > 0]
    prods.sort(key=lambda p: -(p.get("review_count") or 0))
    done = {r["product_id"] for r in read_jsonl("reviews_done.jsonl")}
    todo = [(p, max_pages) for p in prods if p["product_id"] not in done]
    if limit:
        todo = todo[:limit]
    print(f"Reviews: {len(todo)} san pham, {workers} luong")

    def sink(res):
        append_jsonl("reviews.jsonl", res["rows"])
        append_jsonl("reviews_done.jsonl", [{"product_id": res["product_id"]}])
    run_parallel(todo, fetch_reviews, sink, workers, "reviews")



def step_export():
    import pandas as pd
    listing = pd.DataFrame(unique_products())
    detail = pd.DataFrame([{k: v for k, v in r.items() if k != "specs"}
                           for r in read_jsonl("detail.jsonl") if r.get("ok")])
    products = listing.merge(detail.drop(columns=["ok"], errors="ignore"),
                             on="product_id", how="left")
    specs = pd.DataFrame([{"product_id": r["product_id"], **s}
                          for r in read_jsonl("detail.jsonl") if r.get("ok")
                          for s in r.get("specs", [])])
    sellers = pd.DataFrame([r for r in read_jsonl("sellers.jsonl") if r.get("ok")])
    reviews = pd.DataFrame(read_jsonl("reviews.jsonl"))
    if not reviews.empty:
        reviews = reviews.drop_duplicates("review_id")
        reviews["created_at"] = pd.to_datetime(reviews["created_at"], unit="s", errors="coerce")
    for name, df in [("products", products), ("product_specs", specs),
                     ("sellers", sellers), ("reviews", reviews)]:
        out = f"tiki_{name}_raw.csv"
        df.to_csv(out, index=False, encoding="utf-8-sig")
        print(f"{out}: {df.shape}")
    print("\n--- Kiem tra nhanh ---")
    print("Ti le quantity_sold == 0 :", round((products["quantity_sold"].fillna(0) == 0).mean(), 3))
    print("Ti le review_count == 0  :", round((products["review_count"].fillna(0) == 0).mean(), 3))
    print("So danh muc la          :", products["leaf_category"].nunique())
    if not sellers.empty:
        print("Cot seller co gia tri   :", sellers.notna().sum().loc[lambda s: s > 0].to_dict())



def _brief(d, maxlen=80):
    """In gon: moi key mot dong, gia tri dai bi cat, list/dict chi in kich thuoc."""
    if not isinstance(d, dict):
        print(d); return
    for k, v in d.items():
        if isinstance(v, (dict, list)):
            print(f"  {k}: <{type(v).__name__} len={len(v)}>")
        else:
            print(f"  {k}: {str(v)[:maxlen]}")


def step_probe():
    print("== 1. LISTING (1 san pham) ==")
    js = get_json("https://tiki.vn/api/v2/products", {"limit": 2, "category": 1789, "page": 1,
                                                       "sort": "top_seller"})
    it = ((js or {}).get("data") or [None])[0]
    if not it:
        print("KHONG LAY DUOC"); return
    _brief(it)
    print("  quantity_sold =", it.get("quantity_sold"))
    pid, sid = it["id"], it.get("seller_id")
    m = re.search(r"spid=(\d+)", it.get("url_path") or "")
    spid = it.get("seller_product_id") or (m.group(1) if m else None)

    print("\n== 2. DETAIL ==")
    d = get_json(f"https://tiki.vn/api/v2/products/{pid}", {"platform": "web", "spid": spid})
    if d:
        _brief(d)
        print("  current_seller keys:", list((d.get("current_seller") or {}).keys()))
        print("  categories:", d.get("categories"))
    else:
        print("KHONG LAY DUOC")

    print("\n== 3. SELLER WIDGET ==")
    sw = get_json("https://api.tiki.vn/product-detail/v2/widgets/seller",
                  {"seller_id": sid, "mpid": pid, "spid": spid})
    if sw:
        print("  top-level keys:", list(sw.keys()))
        data = sw.get("data") or {}
        print("  data keys:", list(data.keys()))
        _brief(data.get("seller") or {})
    else:
        print("KHONG LAY DUOC")

    print("\n== 4. REVIEWS ==")
    r = get_json("https://tiki.vn/api/v2/reviews", {"product_id": pid, "limit": 2, "page": 1,
                                                    "sort": "score|desc,id|desc,stars|all"})
    if r:
        print("  top-level keys:", list(r.keys()), "| paging:", r.get("paging"))
        rv = (r.get("data") or [None])[0]
        if rv:
            _brief(rv)
            print("  created_by:", rv.get("created_by"))
    else:
        print("KHONG LAY DUOC")


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["probe", "listing", "detail", "sellers", "reviews", "export", "all"])
    ap.add_argument("--pages", type=int, default=20, help="so trang listing / (danh muc, sap xep); 48 sp/trang")
    ap.add_argument("--max-pages", type=int, default=5, help="so trang review / san pham; 20 review/trang")
    ap.add_argument("--limit", type=int, default=0, help="gioi han so muc (detail/sellers/reviews) de chay thu")
    ap.add_argument("--workers", type=int, default=4, help="so luong chay song song (3-4 la vua; cao hon de bi chan)")
    ap.add_argument("--newest-pages", type=int, default=NEWEST_PAGES, help="so trang cho sap xep 'newest'")
    a = ap.parse_args(argv)
    if a.cmd == "probe":
        step_probe()
    if a.cmd in ("listing", "all"):
        step_listing(a.pages, 0, a.newest_pages)
    if a.cmd in ("detail", "all"):
        step_detail(a.limit, a.workers)
    if a.cmd in ("sellers", "all"):
        step_sellers(a.limit, a.workers)
    if a.cmd in ("reviews", "all"):
        step_reviews(a.max_pages, a.limit, a.workers)
    if a.cmd in ("export", "all"):
        step_export()


if __name__ == "__main__":
    import sys
    if "ipykernel" in sys.argv[0]:
        print('Dang chay trong notebook. Hay goi: main(["probe"])')
    else:
        main()

Dang chay trong notebook. Hay goi: main(["probe"])


In [ ]:
main(["listing", "--pages", "1", "--limit", "1"])